In [5]:
import matplotlib.pyplot as plt
from algorithms.tss_file import TSSFile
import os
import json
from pathlib import Path

In [6]:
def find_file(filename, search_dir):
    for root, dirs, files in os.walk(search_dir):
        if filename in files:
            return root
    return None

def glob_files(base_path, file_type):
    paths = Path(base_path).rglob(file_type)
    parent_folders = []
    file_names = []

    for p in paths:
        parent_folders.append(p.parent)
        file_names.append(p.name)

    return parent_folders, file_names

In [7]:
input_folders = [
    r'Y:\TESTDATA\DRDSON\2025\07',
]
folder = r'C:\Users\FeiyuanYang\OneDrive - Cambridge GaN Devices Ltd\Documents\CGD documents\Projects\R0003\data_analysis\R0003 BDS Dynamic Rdson Plots'

In [8]:
for input_folder in input_folders:
    json_folders, json_files = glob_files(input_folder, '*.json')
    for i in range(len(json_files)):
        json_file_path = os.path.join(json_folders[i], json_files[i])
        with open(json_file_path, 'r') as file:
            json_data = json.load(file)
        time_stamp = json_files[i][3:18]
        tss_file = 'Tek' + time_stamp + '.tss'
        tss_folder = find_file(tss_file, json_folders[i])
        if tss_folder is None:
            continue

        chip_id = json_data['chip_id']
        vhv = json_data['high_voltage']
        comment = json_data['comment']

        tss = TSSFile(tss_folder, tss_file, source='Local')
        channel_labels = tss.channel_labels
        num = len(channel_labels.keys())

        path = Path(os.path.join(folder, f'{chip_id}_Rdson_plots'))
        path.mkdir(parents=True, exist_ok=True)

        fig, axs = plt.subplots(num, 1, sharex=True, figsize=(10, 12))
        fig.suptitle(f'{chip_id}_Vhv{vhv}_{comment}_{time_stamp}')
        fig.tight_layout()
        fig.subplots_adjust(hspace=0)
        time1 = tss.waveforms[list(channel_labels.keys())[0]].time_for_frame()*1e9

        index = 0
        for i in [0, 1, 5, 10, 100, 1000, 2000, 2999]:
            index = 0
            for channel in channel_labels.keys():
                values = tss.waveforms[channel].values_for_frame(i)
                axs[index].plot(time1, values, label=i)
                axs[index].set_ylabel(channel_labels[channel])
                axs[index].grid(True)
                index += 1

        plt.legend()
        plt.xlabel('Time (ns)')
        plt.savefig(os.path.join(path, f'WFM_{chip_id}_Vhv{vhv}_{comment}_{time_stamp}.png'))
        plt.close()